# Chapter 10: Multi-Model Router

**Build a request classifier that routes questions to different model tiers based on complexity.**

Not every request needs the most powerful (and expensive) model. A simple FAQ lookup doesn't
require the same reasoning power as a competitive analysis. In this notebook, we build a
routing system that classifies incoming requests by complexity and sends them to the
appropriate model tier — measuring the cost savings compared to using the best model for everything.

**Key insight**: Model tiering saves 60–80% with minimal quality impact.

---

## Setup

Install dependencies and configure API keys. The OpenAI key is only needed if you enable
the optional LLM-based classifier at the end of the notebook.

In [ ]:
# Install dependencies (run once)
!pip install -q openai tabulate

In [ ]:
import os
import re
from dataclasses import dataclass, field
from typing import Literal
from tabulate import tabulate

# Set your API key as an environment variable before running.
# In Colab: use the Secrets sidebar (key icon) or:
#   import os; os.environ["OPENAI_API_KEY"] = "sk-..."
# The key is only required for the optional LLM-based classifier section.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

print("Setup complete.")

## 1. Define Model Pricing

We use Anthropic's Claude model family as our tier example. Prices are per 1 million tokens.

In [ ]:
# Pricing per 1M tokens (USD)
MODEL_PRICING = {
    "haiku": {"input": 0.25, "output": 1.25},
    "sonnet": {"input": 3.00, "output": 15.00},
    "opus": {"input": 15.00, "output": 75.00},
}

# Quick helper: cost for a single request given token counts
def calc_request_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Return the cost in USD for a single request."""
    pricing = MODEL_PRICING[model]
    cost = (input_tokens / 1_000_000) * pricing["input"] + \
           (output_tokens / 1_000_000) * pricing["output"]
    return cost

# Example: a 500-token input / 200-token output request on each model
for model in MODEL_PRICING:
    cost = calc_request_cost(model, 500, 200)
    print(f"{model:>6s}: ${cost:.6f} per request")

## 2. Sample Requests

We create 20 requests across three complexity levels to test our router.

In [ ]:
# 20 sample requests with ground-truth complexity labels
SAMPLE_REQUESTS = [
    # ---------- Simple (10) ----------
    {"query": "What time does the office close?", "expected": "simple"},
    {"query": "What is the Wi-Fi password for guests?", "expected": "simple"},
    {"query": "Where do I submit my expense report?", "expected": "simple"},
    {"query": "How do I reset my email password?", "expected": "simple"},
    {"query": "What holidays does the company observe?", "expected": "simple"},
    {"query": "Who is the head of the marketing department?", "expected": "simple"},
    {"query": "What floor is the cafeteria on?", "expected": "simple"},
    {"query": "Is there parking validation available?", "expected": "simple"},
    {"query": "What is the dress code policy?", "expected": "simple"},
    {"query": "When is the next all-hands meeting?", "expected": "simple"},

    # ---------- Medium (7) ----------
    {"query": "Summarize this quarterly revenue report and highlight the top three trends.", "expected": "medium"},
    {"query": "Draft an email to the team explaining the new PTO policy changes.", "expected": "medium"},
    {"query": "Review this support ticket and suggest a resolution based on our knowledge base.", "expected": "medium"},
    {"query": "List the pros and cons of migrating our CI/CD pipeline to GitHub Actions.", "expected": "medium"},
    {"query": "Write a summary of the customer feedback from last month's NPS survey.", "expected": "medium"},
    {"query": "Explain the differences between our Standard and Premium service tiers.", "expected": "medium"},
    {"query": "Generate a project status update based on these three Jira tickets.", "expected": "medium"},

    # ---------- Complex (3) ----------
    {"query": "Analyze the competitive implications of Acme Corp's new pricing strategy on our market share, considering their recent acquisitions and our current positioning in the enterprise segment.", "expected": "complex"},
    {"query": "Compare our three-year product roadmap against emerging industry trends, identify strategic gaps, and recommend prioritization changes with supporting rationale.", "expected": "complex"},
    {"query": "Evaluate the risk-adjusted ROI of building an in-house LLM fine-tuning capability versus using a managed API, factoring in talent acquisition costs, infrastructure, maintenance, and opportunity cost.", "expected": "complex"},
]

print(f"Total requests: {len(SAMPLE_REQUESTS)}")
for level in ["simple", "medium", "complex"]:
    count = sum(1 for r in SAMPLE_REQUESTS if r["expected"] == level)
    print(f"  {level}: {count}")

## 3. Build the RequestRouter

The router uses simple heuristics to classify request complexity:
- **Query length**: longer queries tend to be more complex.
- **Keyword signals**: words like *analyze*, *compare*, *implications*, *evaluate*, *strategic* suggest deeper reasoning.
- **Clause count**: more commas and conjunctions suggest multi-part requests.

This heuristic approach requires zero API calls and adds negligible latency.

In [ ]:
@dataclass
class RequestRouter:
    """Routes incoming requests to the cheapest model that can handle them."""

    # Tier → model mapping
    tier_model_map: dict = field(default_factory=lambda: {
        "simple": "haiku",
        "medium": "sonnet",
        "complex": "opus",
    })

    # Keywords that signal higher complexity
    COMPLEX_KEYWORDS: list = field(default_factory=lambda: [
        "analyze", "analyse", "compare", "implications", "evaluate",
        "strategic", "risk-adjusted", "prioritization", "recommend",
        "trade-off", "tradeoff", "roi", "cost-benefit", "versus",
    ])
    MEDIUM_KEYWORDS: list = field(default_factory=lambda: [
        "summarize", "summarise", "draft", "review", "explain",
        "list", "generate", "write", "pros and cons", "differences",
    ])

    def classify_complexity(self, query: str) -> Literal["simple", "medium", "complex"]:
        """Classify a query into simple / medium / complex using heuristics."""
        q_lower = query.lower()
        word_count = len(query.split())
        comma_count = query.count(",")

        # Score the query — higher score = more complex
        score = 0

        # Length signals
        if word_count > 30:
            score += 3
        elif word_count > 15:
            score += 1

        # Clause complexity
        if comma_count >= 3:
            score += 2
        elif comma_count >= 1:
            score += 1

        # Keyword signals
        for kw in self.COMPLEX_KEYWORDS:
            if kw in q_lower:
                score += 2
        for kw in self.MEDIUM_KEYWORDS:
            if kw in q_lower:
                score += 1

        # Map score to tier
        if score >= 5:
            return "complex"
        elif score >= 2:
            return "medium"
        else:
            return "simple"

    def route(self, query: str) -> str:
        """Return the model name for the given query."""
        tier = self.classify_complexity(query)
        return self.tier_model_map[tier]


router = RequestRouter()
print("RequestRouter ready.")

## 4. Route All 20 Requests

We run every sample request through the router and check if the heuristic classification
matches the expected (ground-truth) tier.

In [ ]:
# Route every request and collect results
results = []
for req in SAMPLE_REQUESTS:
    query = req["query"]
    expected = req["expected"]
    predicted_tier = router.classify_complexity(query)
    model = router.tier_model_map[predicted_tier]
    match = "yes" if predicted_tier == expected else "NO"
    results.append({
        "query": query[:60] + ("..." if len(query) > 60 else ""),
        "expected": expected,
        "predicted": predicted_tier,
        "model": model,
        "match": match,
    })

# Display routing decisions
headers = ["Query", "Expected", "Predicted", "Model", "Match"]
rows = [[r["query"], r["expected"], r["predicted"], r["model"], r["match"]] for r in results]
print(tabulate(rows, headers=headers, tablefmt="github"))

# Accuracy
correct = sum(1 for r in results if r["match"] == "yes")
print(f"\nRouting accuracy: {correct}/{len(results)} ({100*correct/len(results):.0f}%)")

## 5. Cost Comparison

We calculate total cost under two strategies:
1. **Baseline** — send every request to `opus` (the most capable, most expensive model).
2. **Tiered routing** — send each request to the model selected by the router.

We assume 500 input tokens and 200 output tokens per request (a reasonable average for
enterprise Q&A workloads).

In [ ]:
# Assume uniform token counts for this demo
INPUT_TOKENS = 500
OUTPUT_TOKENS = 200

# Strategy 1: all opus
baseline_cost = sum(
    calc_request_cost("opus", INPUT_TOKENS, OUTPUT_TOKENS)
    for _ in SAMPLE_REQUESTS
)

# Strategy 2: tiered routing
tiered_cost = 0
tier_counts = {"haiku": 0, "sonnet": 0, "opus": 0}
for req in SAMPLE_REQUESTS:
    model = router.route(req["query"])
    tiered_cost += calc_request_cost(model, INPUT_TOKENS, OUTPUT_TOKENS)
    tier_counts[model] += 1

savings = baseline_cost - tiered_cost
savings_pct = (savings / baseline_cost) * 100 if baseline_cost > 0 else 0

print("=== Cost Comparison (20 requests) ===")
print(f"Baseline (all opus):  ${baseline_cost:.6f}")
print(f"Tiered routing:       ${tiered_cost:.6f}")
print(f"Savings:              ${savings:.6f} ({savings_pct:.1f}%)")
print(f"\nTier distribution: {tier_counts}")

# Scale to 10,000 requests/day for a realistic projection
SCALE_FACTOR = 10_000 / len(SAMPLE_REQUESTS)
print("\n=== Projected at 10,000 requests/day ===")
daily_baseline = baseline_cost * SCALE_FACTOR
daily_tiered = tiered_cost * SCALE_FACTOR
monthly_baseline = daily_baseline * 30
monthly_tiered = daily_tiered * 30

comparison = [
    ["All Opus (baseline)", f"${daily_baseline:.2f}", f"${monthly_baseline:.2f}", "—", "—"],
    ["Tiered Routing", f"${daily_tiered:.2f}", f"${monthly_tiered:.2f}",
     f"${(monthly_baseline - monthly_tiered):.2f}", f"{savings_pct:.1f}%"],
]
print(tabulate(comparison,
               headers=["Strategy", "Daily Cost", "Monthly Cost", "Monthly Savings", "Savings %"],
               tablefmt="github"))

## 6. Fallback / Escalation Mechanism

What if the cheap model produces a response that's too short or clearly insufficient?
We add a fallback rule: if the response is below a minimum token threshold, we escalate
to the next tier and re-run.

This is simulated below (we don't call real models, but the logic is production-ready).

In [ ]:
import random

# Tier escalation order
ESCALATION_PATH = ["haiku", "sonnet", "opus"]

def simulate_response_length(model: str, complexity: str) -> int:
    """Simulate output token count. Cheap models produce shorter responses for complex queries."""
    if complexity == "complex" and model == "haiku":
        return random.randint(15, 40)   # often too short
    elif complexity == "complex" and model == "sonnet":
        return random.randint(80, 200)
    else:
        return random.randint(60, 250)


def route_with_fallback(query: str, expected_complexity: str,
                        min_output_tokens: int = 50) -> dict:
    """Route a request, and escalate if the response is too short."""
    initial_model = router.route(query)
    current_idx = ESCALATION_PATH.index(initial_model)
    total_cost = 0
    attempts = []

    while current_idx < len(ESCALATION_PATH):
        model = ESCALATION_PATH[current_idx]
        output_tokens = simulate_response_length(model, expected_complexity)
        cost = calc_request_cost(model, INPUT_TOKENS, output_tokens)
        total_cost += cost
        attempts.append({"model": model, "output_tokens": output_tokens, "cost": cost})

        if output_tokens >= min_output_tokens:
            # Response is sufficient
            break
        else:
            # Escalate
            current_idx += 1

    return {
        "query": query[:50],
        "final_model": attempts[-1]["model"],
        "attempts": len(attempts),
        "total_cost": total_cost,
        "trace": attempts,
    }


# Run fallback routing on a few examples
random.seed(42)
print("=== Fallback / Escalation Demo ===\n")
demo_requests = [
    SAMPLE_REQUESTS[0],   # simple
    SAMPLE_REQUESTS[10],  # medium
    SAMPLE_REQUESTS[17],  # complex
]

for req in demo_requests:
    result = route_with_fallback(req["query"], req["expected"])
    print(f"Query: {result['query']}...")
    print(f"  Attempts: {result['attempts']}, Final model: {result['final_model']}")
    for i, attempt in enumerate(result["trace"]):
        status = "ESCALATED" if attempt["output_tokens"] < 50 else "accepted"
        print(f"    Step {i+1}: {attempt['model']} → {attempt['output_tokens']} tokens ({status})")
    print(f"  Total cost: ${result['total_cost']:.6f}\n")

## 7. (Optional) LLM-Based Classifier

For production systems, you can replace the heuristic classifier with a lightweight LLM
call. A fast, cheap model (like `gpt-4o-mini`) classifies the request, then the expensive
model handles the actual work. The classification call costs fractions of a cent.

**Requires `OPENAI_API_KEY` to be set.**

In [ ]:
# Optional: LLM-based classification
# Uncomment and run if you have an OpenAI API key set.

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    def llm_classify(query: str) -> str:
        """Use a cheap LLM to classify request complexity."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": (
                    "Classify the following user query as exactly one of: "
                    "simple, medium, complex. Respond with only that one word."
                )},
                {"role": "user", "content": query},
            ],
            max_tokens=5,
            temperature=0,
        )
        label = response.choices[0].message.content.strip().lower()
        if label not in ("simple", "medium", "complex"):
            label = "medium"  # safe fallback
        return label

    # Test on a few examples
    for req in [SAMPLE_REQUESTS[0], SAMPLE_REQUESTS[10], SAMPLE_REQUESTS[17]]:
        label = llm_classify(req["query"])
        print(f"[{label:>7s}] {req['query'][:70]}...")
else:
    print("Skipping LLM classifier — OPENAI_API_KEY not set.")
    print("Set it via: os.environ['OPENAI_API_KEY'] = 'sk-...'")

## Key Takeaways

1. **Not every request needs your most powerful model.** A tiered routing strategy can
   cut costs by 60–80% while maintaining quality where it matters.

2. **Heuristic classifiers are a great starting point.** Simple rules (query length,
   keyword detection) get you most of the way. You can graduate to LLM-based classification
   later.

3. **Always build in escalation.** If a cheap model's response is insufficient, automatically
   retry with the next tier up. The extra cost of occasional escalation is far less than
   routing everything to the top tier.

4. **Measure and iterate.** Track routing accuracy and user satisfaction over time. Adjust
   thresholds and keywords as your workload evolves.

---

*From "AI Enterprise Architect" — Chapter 10: Multi-Model Strategies*